# **Lesson_10.1**

## In this lecture

* Update fork

* Logistic regression for the classification task (Part I)

---

### Warming-up exercise
In a radom list (make it up) of numbers, check if there are zeroes and move them to the end of list. Suggest at least two solutions.

In [ ]:
# Write your code here ...

In [ ]:
# Write your code here ...

## Learning outcomes

* Logistic regression for classification task (Part I)

	* Background
	
	* Australia weather prediction

---

In this tutorial, we'll use _logistic regression_, which is better suited for _classification_ problems like predicting whether it will rain tomorrow. Identifying whether a given problem is a _classfication_ or _regression_ problem is an important first step in machine learning.

### Topics covered in this walk-through code-along project
- Downloading a real-world dataset from Kaggle

- Exploratory data analysis and visualization
- Splitting a dataset into training, validation & test sets
- Filling/imputing missing values in numeric columns
- Scaling numeric features to a $(0,1)$ range
- Encoding categorical columns as one-hot vectors
- Training a logistic regression model using Scikit-learn
- Evaluating a model using a validation set and test set
- Saving a model to disk and loading it back


### Classification Problems - background
* Problems where each output must be assigned a discrete category (also called **label** or **class**) are known as **classification problems**

* Classification problems can be binary (yes/no) or multiclass (picking one of many classes)

### Linear Regression for Solving Regression Problems

Linear regression is a commonly used technique for solving regression problems. In a linear regression model, the target is modeled as a linear combination (or weighted sum) of input features. The predictions from the model are evaluated using a loss function like the Root Mean Squared Error (RMSE).


Here's a visual summary of how a linear regression model is structured:
<p align="center">
	<img src="../assets/img/logistic_regr_1.jpg" width="600">
</p>

### Logistic Regression for Solving Classification Problems

Logistic regression is a commonly used technique for solving binary classification problems. In a logistic regression model: 

- we take linear combination (or weighted sum of the input features)

- we apply the sigmoid function to the result to obtain a number between 0 and 1
- this number represents the probability of the input being classified as "Yes"
- instead of RMSE, the cross entropy loss function is used to evaluate the results


Here's a visual summary of how a logistic regression model is structured ([source](http://datahacker.rs/005-pytorch-logistic-regression-in-pytorch/)):

<p align="center">
	<img src="../assets/img/logistic_regr_2.jpg" width="600">
</p>

The sigmoid function applied to the linear combination of inputs has the following formula:

<p align="center">
	<img src="../assets/img/logistic_regr_3.jpg" width="600">
</p>


The output of the sigmoid function is called a logistic, hence the name **logistic regression**.


---

## Australia weather prediction

### Business objectives

### Notebook input

### Notebook output

### Notes / comments

### Notebook objectives

## Import modules

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder

### Config some settings in visualisation libraries

In [ ]:
sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

---

## Load data

In [ ]:
data_path = r'../datasets/weatherAUS.csv'

In [ ]:
raw_df = pd.read_csv(data_path)

In [ ]:
type(raw_df)

In [ ]:
raw_df.info()

In [ ]:
raw_df.isna().sum()

In [ ]:
raw_df.head()

* While we should be able to fill in missing values for most columns, it might be a good idea to discard the rows where the value of `RainTomorrow` or `RainToday` is missing to make our analysis and modeling simpler (since one of them is the target variable, and the other is likely to be very closely related to the target variable). 

In [ ]:
raw_df.dropna(subset=['RainToday', 'RainTomorrow'], inplace=True)

---

## EDA

* Before training a machine learning model, its always a must to explore the distributions of various columns and see how they are related to the target column. Let's explore and visualize the data using the Plotly, Matplotlib and Seaborn libraries

In [ ]:
px.histogram(raw_df, x='Location', title='Location vs. Rainy Days', color='RainToday')

In [ ]:
px.histogram(raw_df, 
             x='Temp3pm', 
             title='Temperature at 3 pm vs. Rain Tomorrow', 
             color='RainTomorrow')

In [ ]:
px.histogram(raw_df, 
             x='RainTomorrow', 
             color='RainToday', 
             title='Rain Tomorrow vs. Rain Today')

In [ ]:
px.scatter(raw_df.sample(2000), 
           title='Min Temp. vs Max Temp.',
           x='MinTemp', 
           y='MaxTemp', 
           color='RainToday')

In [ ]:
px.scatter(raw_df.sample(2000), 
           title='Temp (3 pm) vs. Humidity (3 pm)',
           x='Temp3pm',
           y='Humidity3pm',
           color='RainTomorrow')

### Working with sample (FYI - optional)

* Sometimes it is suitable to work with a fraction of a dataset if, for instance the dataset is excessively large and it slows everything down. You can take a sample of your dataset

In [ ]:
sample_df = raw_df.sample(frac=0.1).copy()

In [ ]:
sample_df.info()

## Training, Validation and Test Sets

While building real-world machine learning models, it is quite common to split the dataset into three parts:

1. **Training set** - used to train the model, i.e., compute the loss and adjust the model's weights using an optimization technique. 


2. **Validation set** - used to evaluate the model during training, tune model hyperparameters (optimization technique, regularization etc.), and pick the best version of the model. Picking a good validation set is essential for training models that generalize well.


3. **Test set** - used to compare different models or approaches and report the model's final accuracy. For many datasets, test sets are provided separately. The test set should reflect the kind of data the model will encounter in the real-world, as closely as feasible.

<p align="center">
	<img src="../assets/img/dataset_splitting.jpg" width="480">
</p>

* As a general rule of thumb you can use around 60-70% of the data for the **training set**, 20% for the **validation** set and 10-20% for the **test set**. If a separate test set is already provided, you can use a 75%-25% training-validation split

* When rows in the dataset have no inherent order, it's common practice to pick random subsets of rows for creating test and validation sets. This can be done using the `train_test_split` utility from `scikit-learn`

In [ ]:
train_val_df, test_df = train_test_split(raw_df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

In [ ]:
print('train_df.shape :', train_df.shape)
print('val_df.shape :', val_df.shape)
print('test_df.shape :', test_df.shape)

* <u>However</u>, while working with dates, it's often a better idea to separate the training, validation and test sets with time, so that the model is trained on data from the past and evaluated on data from the future
* For the current dataset, we can use the Date column in the dataset to create another column for year. We'll pick the last two years for the test set, and one year before it for the validation set

In [ ]:
plt.title('No. of Rows per Year')
sns.countplot(x=pd.to_datetime(raw_df['Date']).dt.year);

In [ ]:
year = pd.to_datetime(raw_df['Date']).dt.year

train_df = raw_df[year < 2015]
val_df = raw_df[year == 2015]
test_df = raw_df[year > 2015]

In [ ]:
print('train_df.shape :', train_df.shape)
print('val_df.shape :', val_df.shape)
print('test_df.shape :', test_df.shape)

* While not a perfect 60-20-20 split, we have ensured that the test validation and test sets both contain data for all 12 months of the year

In [ ]:
train_df.head()

In [ ]:
val_df.head()

In [ ]:
test_df.head()

## Identifying Input and Target Columns
* Often, not all the columns in a dataset are useful for training a model. In the current dataset, we can ignore the `Date` column, since we only want to weather conditions to make a prediction about whether it will rain the next day
* Let's create a list of input columns, and also identify the target column

In [ ]:
input_cols = list(train_df.columns)[1:-1]
target_col = 'RainTomorrow'

In [ ]:
input_cols

* We can now create inputs and targets for the training, validation and test sets for further processing and model training

In [ ]:
train_inputs = train_df[input_cols].copy()
train_targets = train_df[target_col].copy()

val_inputs = val_df[input_cols].copy()
val_targets = val_df[target_col].copy()

test_inputs = test_df[input_cols].copy()
test_targets = test_df[target_col].copy()

In [ ]:
test_inputs.head()

In [ ]:
test_targets.head()

* Let's also identify which of the columns are numerical and which ones are categorical. This will be useful later, as we'll need to convert the categorical data to numbers for training a logistic regression model

In [ ]:
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

In [ ]:
train_inputs[numeric_cols].describe()

* Do the ranges of the numeric columns seem reasonable? If not, we may have to do some data cleaning as well

* Let's also check the number of categories in each of the categorical columns

In [ ]:
train_inputs[categorical_cols].nunique()

---

##### Reminder: do not forget to **Clear All Outputs**
### Now you can commit and push your code to **GitHub**